# Phân tích lý thuyết: Phát hiện ảnh trùng lặp bằng pHash

## 1. Hàm Băm pHash:

Trong bước phân tích thống kê tập dữ liệu, một nhiệm vụ quan trọng là xác định các ảnh bị trùng lặp hoặc gần như trùng lặp. Để thực hiện việc này, chúng ta sẽ sử dụng một kỹ thuật gọi là **Perceptual Hashing**, cụ thể là thuật toán **pHash**.

### 1.1. pHash là gì?

pHash là một thuật toán tạo ra một dấu vân tay kỹ thuật số - digital fingerprint cho một hình ảnh, dựa trên nội dung trực quan của nó. Điểm khác biệt cốt lõi của pHash so với các hàm băm mật mã (ví dụ: MD5, SHA-256) nằm ở mục đích sử dụng:

- **Hàm băm mật mã:** Dùng để kiểm tra sự toàn vẹn tuyệt đối. Thay đổi dù chỉ 1 pixel cũng sẽ tạo ra một mã băm hoàn toàn khác biệt.
- **Hàm băm pHash:** Dùng để kiểm tra sự tương đồng cảm nhận. Những thay đổi nhỏ, không ảnh hưởng đến nội dung chính của ảnh như thay đổi kích thước, nén file, chỉnh sáng nhẹ sẽ chỉ tạo ra một mã băm gần giống với mã băm gốc.

> **Nguyên tắc của pHash:** Những hình ảnh trông giống nhau sẽ có mã băm gần giống nhau.

### 1.2. Thuật toán pHash hoạt động như thế nào?

pHash không phải là một công thức toán học đơn lẻ, mà là một quy trình xử lý ảnh gồm nhiều bước. Phiên bản phổ biến nhất dựa trên biến đổi Cosine rời rạc (Discrete Cosine Transform - DCT) hoạt động như sau:

1.  **Giảm kích thước:** Ảnh được thu nhỏ về kích thước cố định rất nhỏ ví dụ như 32x32 pixels.
    *   *Mục đích:* Loại bỏ các chi tiết phức tạp và sự phụ thuộc vào độ phân giải gốc. Bước này buộc thuật toán phải tập trung vào cấu trúc tổng thể của ảnh.

2.  **Chuyển sang ảnh xám:** Ảnh được chuyển đổi sang thang độ xám (grayscale).
    *   *Mục đích:* Loại bỏ thông tin màu sắc, chỉ giữ lại thông tin về cường độ sáng. Điều này làm cho thuật toán trở nên bền vững trước các thay đổi về tông màu.

3.  **Tính toán DCT (Discrete Cosine Transform):** Áp dụng phép biến đổi DCT lên ma trận pixel 32x32.
    *   *Mục đích:* DCT là một kỹ thuật toán học giúp tách ảnh thành các thành phần tần số khác nhau. Các tần số thấp (đại diện cho cấu trúc chính, hình dạng lớn, sự thay đổi ánh sáng từ từ) sẽ được tập trung ở góc trên bên trái của ma trận kết quả.

4.  **Chắt lọc DCT:** Chỉ giữ lại một phần nhỏ (thường là 8x8) ở góc trên bên trái của ma trận DCT.
    *   *Mục đích:* Đây là bước chắt lọc thông tin quan trọng nhất. Chúng ta chỉ lấy 64 giá trị đại diện cho bản chất cốt lõi của ảnh và loại bỏ hết các chi tiết không cần thiết.

5.  **Tính giá trị trung bình:** Tính giá trị trung bình của 64 giá trị trong ma trận 8x8 vừa thu được.

6.  **Tạo mã băm nhị phân:** So sánh tuần tự 64 giá trị DCT với giá trị trung bình.
    *   Nếu một giá trị **lớn hơn hoặc bằng** trung bình, gán bit **`1`**.
    *   Nếu một giá trị **nhỏ hơn** trung bình, gán bit **`0`**.
    *   Kết quả là một chuỗi 64-bit (ví dụ: `11010010...`), đây chính là **mã pHash** của ảnh.

### 1.3. Cơ chế so sánh: Khoảng cách Hamming

pHash không so sánh trực tiếp hai ảnh. Thay vào đó, nó so sánh hai fingerprint 64-bit của chúng bằng một phép đo gọi là **Khoảng cách Hamming - Hamming Distance**.

> **Định nghĩa:** Khoảng cách Hamming là số lượng các vị trí bit khác nhau giữa hai chuỗi bit có cùng độ dài.

**Ví dụ:**
- Hash A: `10110010`
- Hash B: `11110110`
- So sánh từng bit, ta thấy chúng khác nhau ở vị trí thứ 2 và thứ 6. Do đó, Khoảng cách Hamming là **2**.

Nếu hai ảnh có cấu trúc cảm nhận tương tự nhau, mã pHash của chúng sẽ rất giống nhau, dẫn đến Khoảng cách Hamming rất nhỏ (gần bằng 0).

## 2. Diễn giải kết quả và lựa chọn ngưỡng tương đồng

### 2.1. Phân loại mức độ tương đồng

Dựa vào giá trị Khoảng cách Hamming, chúng ta có thể phân loại mức độ tương đồng giữa hai ảnh thành các cấp độ sau:

*   **Ảnh trùng tặp - Identical / Duplicate:**
    *   **Định nghĩa:** Khi khoảng cách Hamming **chính xác bằng 0**.
    *   **Ý nghĩa:** Chuỗi 64-bit của hai ảnh giống hệt nhau. Đây là hai bản sao của nhau về mặt cảm nhận, ví dụ như copy-paste, đổi tên file.

*   **Ảnh gần giống - Near-Duplicate / Similar:**
    *   **Định nghĩa:** Khi khoảng cách Hamming là một giá trị **nhỏ nhưng lớn hơn 0**.
    *   **Ý nghĩa:** Phần lớn cấu trúc cảm nhận của hai ảnh là như nhau. Một vài bit trong chuỗi băm bị thay đổi do các chỉnh sửa nhỏ như:
        - Thay đổi kích thước.
        - Nén ảnh với chất lượng khác nhau.
        - Thêm một watermark hoặc logo nhỏ.
        - Cropping một phần nhỏ.
        - Chỉnh sửa nhẹ về độ sáng hoặc màu sắc.

*   **Ảnh Khác biệt - Dissimilar:**
    *   **Định nghĩa:** Khi Khoảng cách Hamming là một giá trị **lớn**.
    *   **Ý nghĩa:** Cấu trúc tổng thể và nội dung chính của hai ảnh hoàn toàn khác nhau.

### 2.2. Phân tích về ngưỡng tương đồng - Threshold

Việc lựa chọn một **ngưỡng (threshold)** là cực kỳ quan trọng để tự động hóa việc phân loại. Đây là một sự đánh đổi giữa **độ chính xác (Precision)** và **độ bao phủ (Recall)**.

| Ngưỡng | Độ Chính Xác (Precision) | Độ Bao Phủ (Recall) | Rủi Ro |
| :--- | :--- | :--- | :--- |
| **Thấp (ví dụ: ≤ 3)** | Rất Cao | Thấp | Bỏ sót nhiều cặp gần giống (False Negatives) |
| **Trung bình (ví dụ: 4-7)** | Cao | Trung bình | Cân bằng giữa hai yếu tố |
| **Cao (ví dụ: ≥ 8)** | Thấp | Cao | Bao gồm cả các ảnh khác nhau (False Positives) |

Dựa trên kinh nghiệm và các tài liệu tham khảo, chúng ta có thể phân tích các khoảng ngưỡng như sau:

*   **Ngưỡng an toàn (`Khoảng cách ≤ 5`):**
    *   Đây là vùng có độ tin cậy rất cao. Các cặp ảnh trong khoảng này gần như chắc chắn là biến thể của cùng một ảnh gốc. Đây là ngưỡng khởi đầu được khuyến nghị cho hầu hết các tác vụ làm sạch dữ liệu.

*   **Vùng xám (`6 ≤ Khoảng cách ≤ 10`):**
    *   Đây là vùng có sự mơ hồ. Các cặp ảnh có thể là ảnh gần giống đã qua chỉnh sửa nhiều, hoặc là hai ảnh khác nhau nhưng có bố cục tình cờ tương tự. Việc sử dụng ngưỡng trong khoảng này đòi hỏi phải kiểm tra thủ công một vài mẫu để đánh giá.

*   **Ngưỡng khác biệt (`Khoảng cách > 10`):**
    *   Có thể tự tin kết luận rằng đây là hai ảnh riêng biệt, không liên quan về mặt nội dung.